# Amazon Prime Video Content Analytics & Insights
Exploratory Data Analysis and Visualization of the Amazon Prime Movies and TV Shows dataset.

**Dataset:** `amazon_prime_titles.csv` (Kaggle: shivamb / amazon-prime-movies-and-tv-shows)

**Notebook flow:** load data -> clean data -> 9 visualizations -> key insights -> export cleaned data and a results summary.

## 1. Setup

In [ ]:
import os
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110

# Works whether the notebook is run from the repo root or from notebooks/
ROOT = ".." if os.path.basename(os.getcwd()) == "notebooks" else "."
IMG_DIR = os.path.join(ROOT, "images")
DATA_DIR = os.path.join(ROOT, "data")
DOCS_DIR = os.path.join(ROOT, "docs")
for d in (IMG_DIR, DATA_DIR, DOCS_DIR):
    os.makedirs(d, exist_ok=True)


def save_fig(fig, name):
    fig.tight_layout()
    fig.savefig(os.path.join(IMG_DIR, name), dpi=150, bbox_inches="tight")
    plt.show()

## 2. Load the dataset

In [ ]:
CSV_NAME = "amazon_prime_titles.csv"
candidates = [os.path.join(DATA_DIR, CSV_NAME), CSV_NAME]
csv_path = next((p for p in candidates if os.path.exists(p)), None)

if csv_path is None:
    # Google Colab fallback: ask for the file
    try:
        from google.colab import files
        uploaded = files.upload()
        csv_path = list(uploaded.keys())[0]
    except ImportError:
        raise FileNotFoundError(
            f"Could not find {CSV_NAME}. Download it from Kaggle and put it in the data/ folder."
        )

df_raw = pd.read_csv(csv_path)
df = df_raw.copy()
print("File:", csv_path)
print("Shape:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
print("Content split:")
print(df["type"].value_counts())
print()
print("Release years:", int(df["release_year"].min()), "-", int(df["release_year"].max()))

## 3. Data cleaning

Steps: strip text, inspect missing values, fix misplaced values, drop duplicates, fix data types, flag outliers.

In [ ]:
# 3.1 Strip whitespace in all text columns
text_cols = df.select_dtypes(include="object").columns
for c in text_cols:
    df[c] = df[c].astype("string").str.strip()
    df[c] = df[c].replace("", pd.NA)

# 3.2 Missing values
missing = pd.DataFrame({
    "missing": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(1),
}).sort_values("missing", ascending=False)
missing

In [ ]:
# 3.3 Bad entries: a duration typed into the rating column
bad_mask = df["rating"].astype("string").str.contains(r"\d+\s*(?:min|season)", case=False, na=False)
print("Rows with duration in the rating column:", int(bad_mask.sum()))

move_mask = bad_mask & df["duration"].isna()
df.loc[move_mask, "duration"] = df.loc[move_mask, "rating"]
df.loc[bad_mask, "rating"] = pd.NA

In [ ]:
# 3.4 Duplicates
print("Duplicate rows   :", int(df.duplicated().sum()))
print("Duplicate show_id:", int(df["show_id"].duplicated().sum()))
df = df.drop_duplicates().drop_duplicates(subset="show_id").reset_index(drop=True)
print("Shape after removing duplicates:", df.shape)

In [ ]:
# 3.5 Fill missing text values with "Unknown"
for c in ["director", "cast", "country", "rating", "listed_in", "description"]:
    if c in df.columns:
        df[c] = df[c].fillna("Unknown")

# 3.6 Data types
# date_added is text like "March 30, 2021" -> datetime. Rows without a date are KEPT
# (this dataset has many missing dates); they are only left out of the date-based charts.
df["date_added"] = pd.to_datetime(df["date_added"], errors="coerce")
df["year_added"] = df["date_added"].dt.year
df["month_added"] = df["date_added"].dt.month

# duration is mixed: "113 min" for movies, "2 Seasons" for TV shows
dur = df["duration"].astype("string")
is_movie = df["type"] == "Movie"
is_show = df["type"] == "TV Show"
df["duration_min"] = np.where(
    is_movie, pd.to_numeric(dur.str.extract(r"(\d+)\s*min", expand=False), errors="coerce"), np.nan
)
df["seasons"] = np.where(
    is_show, pd.to_numeric(dur.str.extract(r"(\d+)\s*season", flags=re.I, expand=False), errors="coerce"), np.nan
)

# Helper columns for later analysis
df["primary_country"] = df["country"].str.split(",").str[0].str.strip()
df["primary_genre"] = df["listed_in"].str.split(",").str[0].str.strip()
df.dtypes

In [ ]:
# 3.7 Outliers (IQR rule). Real values, so we keep them and just flag them.
def iqr_flag(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return (s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)

df["duration_outlier"] = iqr_flag(df["duration_min"]).fillna(False)
df["seasons_outlier"] = iqr_flag(df["seasons"]).fillna(False)
print("Movie duration outliers :", int(df["duration_outlier"].sum()))
print("TV seasons outliers     :", int(df["seasons_outlier"].sum()))

In [ ]:
# 3.8 Save the cleaned dataset
cleaned_path = os.path.join(DATA_DIR, "amazon_prime_cleaned.csv")
df.to_csv(cleaned_path, index=False)
print("Saved:", cleaned_path, "| shape:", df.shape)

## 4. Visualizations

### 4.1 Movies vs TV Shows

In [ ]:
type_counts = df["type"].value_counts()
fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.pie(type_counts, labels=type_counts.index, autopct="%1.1f%%", startangle=90,
       colors=sns.color_palette("deep")[:len(type_counts)], wedgeprops={"edgecolor": "white"})
ax.set_title("Amazon Prime: Movies vs TV Shows")
save_fig(fig, "01_movies_vs_tvshows.png")

### 4.2 Titles added per year (stacked by type)

In [ ]:
added = df.dropna(subset=["year_added"]).copy()
added["year_added"] = added["year_added"].astype(int)
by_year = pd.crosstab(added["year_added"], added["type"])

fig, ax = plt.subplots(figsize=(10, 5))
by_year.plot(kind="bar", stacked=True, ax=ax, width=0.85)
ax.set_title(f"Titles added per year (only the {len(added):,} titles with a date_added)")
ax.set_xlabel("Year added")
ax.set_ylabel("Number of titles")
ax.legend(title="Type")
save_fig(fig, "02_titles_added_per_year.png")

### 4.3 Top 10 countries

In [ ]:
countries = (
    df.loc[df["country"] != "Unknown", "country"]
    .str.split(",").explode().str.strip()
)
countries = countries[countries != ""]
top_countries = countries.value_counts().head(10)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(x=top_countries.values, y=top_countries.index, ax=ax, color=sns.color_palette("deep")[0])
ax.set_title("Top 10 countries (titles with a known country)")
ax.set_xlabel("Number of titles")
ax.set_ylabel("")
save_fig(fig, "03_top_countries.png")

### 4.4 Top 10 genres

In [ ]:
genres = df["listed_in"].str.split(",").explode().str.strip()
genres = genres[(genres != "") & (genres != "Unknown")]
top_genres = genres.value_counts().head(10)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(x=top_genres.values, y=top_genres.index, ax=ax, color=sns.color_palette("deep")[1])
ax.set_title("Top 10 genres")
ax.set_xlabel("Number of titles")
ax.set_ylabel("")
save_fig(fig, "04_top_genres.png")

### 4.5 Rating distribution

In [ ]:
rating_counts = df.loc[df["rating"] != "Unknown", "rating"].value_counts().head(15)

fig, ax = plt.subplots(figsize=(9, 5.5))
sns.barplot(x=rating_counts.values, y=rating_counts.index, ax=ax, color=sns.color_palette("deep")[2])
ax.set_title("Rating distribution (top 15, titles with a known rating)")
ax.set_xlabel("Number of titles")
ax.set_ylabel("")
save_fig(fig, "05_rating_distribution.png")

### 4.6 Movie duration

In [ ]:
movie_dur = df["duration_min"].dropna()

fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(movie_dur, bins=40, kde=True, ax=ax, color=sns.color_palette("deep")[3])
ax.axvline(movie_dur.median(), color="black", linestyle="--", label=f"Median: {movie_dur.median():.0f} min")
ax.set_title("Movie duration (minutes)")
ax.set_xlabel("Duration (minutes)")
ax.set_ylabel("Number of movies")
ax.legend()
save_fig(fig, "06_movie_duration.png")

### 4.7 Number of seasons for TV shows

In [ ]:
season_counts = df["seasons"].dropna().astype(int).value_counts().sort_index()

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(x=season_counts.index.astype(str), y=season_counts.values, ax=ax, color=sns.color_palette("deep")[4])
ax.set_title("TV shows by number of seasons")
ax.set_xlabel("Number of seasons")
ax.set_ylabel("Number of TV shows")
save_fig(fig, "07_tv_seasons.png")

### 4.8 Heatmap: year added vs month added

In [ ]:
heat = pd.crosstab(added["year_added"], added["month_added"].astype(int))
heat = heat.reindex(columns=range(1, 13), fill_value=0)
heat.columns = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(heat, cmap="YlGnBu", annot=True, fmt="d", linewidths=0.5, ax=ax, cbar_kws={"label": "Titles added"})
ax.set_title("When were titles added? (year vs month)")
ax.set_xlabel("Month")
ax.set_ylabel("Year added")
save_fig(fig, "08_heatmap_year_month.png")

### 4.9 Titles by release year

In [ ]:
release = df.groupby(["release_year", "type"]).size().unstack(fill_value=0)
release = release[release.index >= 1970]

fig, ax = plt.subplots(figsize=(10, 5))
release.plot(ax=ax, linewidth=2)
ax.set_title("Titles by release year (1970 onwards)")
ax.set_xlabel("Release year")
ax.set_ylabel("Number of titles")
ax.legend(title="Type")
save_fig(fig, "09_titles_by_release_year.png")

## 5. Key insights (computed from the data)

In [ ]:
n = len(df)
movie_pct = (df["type"] == "Movie").mean() * 100
show_pct = (df["type"] == "TV Show").mean() * 100
peak_year = int(by_year.sum(axis=1).idxmax()) if len(by_year) else None
recent_pct = (df["release_year"] >= 2015).mean() * 100
rated = df.loc[df["rating"] != "Unknown", "rating"]
top2_rating = rated.value_counts().head(2)
top2_share = top2_rating.sum() / len(rated) * 100 if len(rated) else float("nan")
one_season_pct = (df.loc[is_show, "seasons"] == 1).sum() / max(is_show.sum(), 1) * 100
date_known_pct = df["date_added"].notna().mean() * 100
country_known_pct = (df["country"] != "Unknown").mean() * 100

insights = [
    f"The catalogue has {n:,} titles: {movie_pct:.1f}% Movies and {show_pct:.1f}% TV Shows.",
    f"{recent_pct:.1f}% of titles were released in 2015 or later.",
    f"Most titles were added in {peak_year} (among titles that have a date_added)." if peak_year else "No date_added data.",
    f"Top countries: {', '.join(top_countries.index[:3])}.",
    f"Top genres: {', '.join(top_genres.index[:3])}.",
    f"The two most common ratings are {', '.join(top2_rating.index)} ({top2_share:.1f}% of rated titles).",
    f"Median movie length is {movie_dur.median():.0f} minutes.",
    f"{one_season_pct:.1f}% of TV shows have only 1 season.",
    f"Data quality: date_added is known for only {date_known_pct:.1f}% of titles and country for {country_known_pct:.1f}%, "
    "so date and country charts describe a subset of the catalogue.",
]
for i in insights:
    print("-", i)

## 6. Export a results summary

In [ ]:
summary_path = os.path.join(DOCS_DIR, "results_summary.md")
with open(summary_path, "w", encoding="utf-8") as f:
    f.write("# Results summary (auto-generated by the notebook)\n\n")
    f.write(f"- Rows x columns (raw): {df_raw.shape[0]:,} x {df_raw.shape[1]}\n")
    f.write(f"- Rows x columns (cleaned): {df.shape[0]:,} x {df.shape[1]}\n")
    f.write(f"- Movie duration outliers (IQR): {int(df['duration_outlier'].sum())}\n")
    f.write(f"- TV seasons outliers (IQR): {int(df['seasons_outlier'].sum())}\n\n")
    f.write("## Key insights\n\n")
    for i in insights:
        f.write(f"- {i}\n")
    f.write("\n## Missing values (raw data)\n\n")
    raw_missing = df_raw.isna().sum()
    for col, val in raw_missing[raw_missing > 0].sort_values(ascending=False).items():
        f.write(f"- {col}: {int(val):,}\n")
print("Saved:", summary_path)

## 7. Conclusion

The notebook turns the raw Amazon Prime catalogue into clear visual insights about its mix of movies and TV shows,
where titles come from, which genres and ratings dominate, and how the catalogue grew. Future scope: build a
content-based recommendation system using genre, cast and description, and compare directly with the Netflix catalogue.